In [ ]:
# =========================================================
# STEP 0 — Essential imports and mixed precision setup
# =========================================================
import os, random, numpy as np, cv2, gc
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import EfficientNetV2L
from tensorflow.keras import mixed_precision

# ✅ Enable mixed precision to halve memory usage
mixed_precision.set_global_policy('mixed_float16')

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

: 

In [ ]:
# =========================================================
# STEP 1 — Load PlantDoc Dataset (299×299 input)
# =========================================================
from sklearn.model_selection import train_test_split

# Use local dataset directory
dataset_root = os.path.join(os.getcwd(), 'dataset', 'archive', 'PlantDoc-Dataset')
train_dir = os.path.join(dataset_root, 'train')
test_dir  = os.path.join(dataset_root, 'test')

def load_plantdoc_data(data_dir, img_size=(299,299)):
    X, y = [], []
    class_names = sorted([d for d in os.listdir(data_dir)
                          if os.path.isdir(os.path.join(data_dir,d))])
    class_to_idx = {cls:i for i,cls in enumerate(class_names)}
    for cls in class_names:
        for img_file in os.listdir(os.path.join(data_dir,cls)):
            path = os.path.join(data_dir,cls,img_file)
            img = cv2.imread(path)
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, img_size)
            X.append(img); y.append(class_to_idx[cls])
    return np.array(X,np.float32)/255.0, np.array(y,np.int32), class_names

X_train, y_train, class_names = load_plantdoc_data(train_dir)
X_test,  y_test,  _           = load_plantdoc_data(test_dir)
num_classes = len(class_names)

# Split into train/validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=SEED
)

# One‑hot encode labels
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat   = to_categorical(y_val, num_classes)
y_test_cat  = to_categorical(y_test, num_classes)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

In [ ]:
# =========================================================
# STEP 2 — Class weights (imbalanced dataset support)
# =========================================================
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(num_classes),
    y=y_train
)
class_weights = {i:w for i,w in enumerate(class_weights)}
print("Class Weights:", class_weights)

In [ ]:
# =========================================================
# STEP 3 — Custom callback for Cohen’s Kappa monitoring
# =========================================================
from sklearn.metrics import cohen_kappa_score
class KappaCallback(tf.keras.callbacks.Callback):
    def __init__(self,X_val,y_val_cat):
        super().__init__()
        self.X_val=X_val; self.y_val_cat=y_val_cat
    def on_epoch_end(self,epoch,logs=None):
        y_pred=np.argmax(self.model.predict(self.X_val,verbose=0),axis=1)
        y_true=np.argmax(self.y_val_cat,axis=1)
        kappa=cohen_kappa_score(y_true,y_pred)
        val_acc=logs.get("val_accuracy")
        print(f"\nEpoch {epoch+1}: Val.Acc {val_acc:.4f} | Kappa {kappa:.4f}")

In [ ]:
# 📦 Install Keras Tuner
!pip install keras-tuner --quiet

In [ ]:
from keras_tuner import HyperModel

In [ ]:
# =========================================================
# STEP 4 — Build EfficientNetV2‑L with tunable hyperparams
# =========================================================
from keras_tuner import HyperModel

class EfficientNetV2LHyperModel(HyperModel):
    def __init__(self,num_classes,input_shape=(299,299,3)):
        self.num_classes=num_classes
        self.input_shape=input_shape

    def build(self,hp):
        # Hyperparameters
        dropout = hp.Choice('dropout',[0.2,0.3,0.4])
        lr = hp.Float('lr',1e-5,1e-3,sampling='log')
        optimizer_choice = hp.Choice('optimizer',['adam','rmsprop','sgd'])
        hp.Choice('batch_size',[8,16,32])

        # Lightweight augmentation
        aug = tf.keras.Sequential([
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1)
        ])

        # Load pretrained EfficientNetV2‑L base
        base = EfficientNetV2L(include_top=False,weights='imagenet',input_shape=self.input_shape)
        for l in base.layers: l.trainable=False
        for l in base.layers[200:350]: l.trainable=True  # mid‑region trainable

        inputs = layers.Input(shape=self.input_shape)
        x = aug(inputs)
        x = base(x,training=False)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dropout(dropout)(x)
        x = layers.Dense(512,activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(dropout)(x)
        outputs = layers.Dense(self.num_classes,activation='softmax',dtype='float32')(x)

        model = models.Model(inputs,outputs)

        # Optimizer mapping
        opt_map = {
            'adam': optimizers.Adam(learning_rate=lr),
            'rmsprop': optimizers.RMSprop(learning_rate=lr),
            'sgd': optimizers.SGD(learning_rate=lr,momentum=0.9)
        }
        model.compile(optimizer=opt_map[optimizer_choice],
                      loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
                      metrics=['accuracy'])
        return model

In [ ]:
# =========================================================
# STEP 5 — Hyperparameter search with Keras Tuner
# =========================================================
import keras_tuner as kt

def model_builder(hp):
    return EfficientNetV2LHyperModel(num_classes).build(hp)

output_dir = os.path.join(os.getcwd(), 'EfficientNetV2LTuner')
os.makedirs(output_dir, exist_ok=True)

tuner = kt.BayesianOptimization(
    model_builder,
    objective='val_accuracy',
    max_trials=5,  # fewer => less RAM and time
    executions_per_trial=1,
    directory=output_dir,
    project_name='effnetv2l_tuning'
)

tuner.search_space_summary()

# Streaming datasets consume less RAM
train_ds = tf.data.Dataset.from_tensor_slices((X_train,y_train_cat)).shuffle(512).batch(8).prefetch(tf.data.AUTOTUNE)
val_ds   = tf.data.Dataset.from_tensor_slices((X_val,y_val_cat)).batch(8).prefetch(tf.data.AUTOTUNE)

tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=3,  # short runs for tuning
    class_weight=class_weights,
    callbacks=[KappaCallback(X_val,y_val_cat)],
    verbose=1
)

best_hp = tuner.get_best_hyperparameters(1)[0]
print("\n🎯 Best Hyperparameters Found:")
for k,v in best_hp.values.items():
    print(f" {k}: {v}")
tuner.results_summary(num_trials=3)

In [ ]:
# =========================================================
# STEP 6 — Build and fully train best model
# =========================================================
tf.keras.backend.clear_session(); gc.collect()

best_model = tuner.hypermodel.build(best_hp)
best_batch = best_hp.get('batch_size')  # 👈 fixed line

output_dir = os.path.join(os.getcwd(), 'EfficientNetV2LTuner')
os.makedirs(output_dir, exist_ok=True)

callbacks_final = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.3, patience=2, min_lr=1e-7),
    tf.keras.callbacks.ModelCheckpoint(os.path.join(output_dir, 'best_effnetv2l.keras'), monitor='val_accuracy',
                                       save_best_only=True, verbose=1),
    KappaCallback(X_val, y_val_cat)
]

history = best_model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=15,
    batch_size=best_batch,
    class_weight=class_weights,
    callbacks=callbacks_final,
    verbose=1
)

best_model.summary()
best_model.save(os.path.join(output_dir, 'final_best_model.keras'))

In [ ]:
# =========================================================
# STEP 7 — Evaluate on test set
# =========================================================
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
model = tf.keras.models.load_model(os.path.join(os.getcwd(), 'EfficientNetV2LTuner', 'final_best_model.keras'))

y_pred_probs = model.predict(X_test,verbose=0)
y_pred = np.argmax(y_pred_probs,axis=1)
y_true = y_test

acc = accuracy_score(y_true,y_pred)
f1w = f1_score(y_true,y_pred,average='weighted')
kappa = cohen_kappa_score(y_true,y_pred)
print(f"\n📊 Test Accuracy: {acc:.4f} | Weighted F1: {f1w:.4f} | Kappa: {kappa:.4f}")

In [ ]:
# =========================================================
# STEP 8 — Quantize model to INT8 TFLite (lightweight deployment)
# =========================================================
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_int8 = converter.convert()

save_path = os.path.join(os.getcwd(), 'EfficientNetV2LTuner', 'effnetv2l_int8.tflite')
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path,"wb") as f: f.write(tflite_int8)
print("✅ INT8 Quantized Model saved at:", save_path)